# ODL Assignment 2 — Task 2
## Model Tuning: hyperparameter search for the multilayer perceptron

**What this notebook does.** Task 1 chose the MLP as the stronger model. Here we tune
its hyperparameters to build the final model, using a pipeline that goes beyond the two
methods in the lecture slides:

1. **Race two searchers** — Random Search (the slides' pick) vs Bayesian Optimisation
   (smarter, learns from past trials) on the wide space, scored with **k-fold cross-validation**.
2. **Importance analysis** — which hyperparameters actually matter.
3. **Focus + verification** — narrow to the influential knobs, confirm the fixed ones.
4. **Final model** — retrain the best config and evaluate on the sealed test set.

> The optimiser type stays **Nadam** (fixed in Task 1); only its learning rate and, for
> the importance check, its beta/epsilon terms enter the search.

> ⚠️ The searches train hundreds of models across the folds, so the full run takes tens
> of minutes. Trial counts are set modestly here; the report used 28 trials per searcher.

---
## Step 1 — Imports and data

The development set is `train_processed.csv` + `val_processed.csv` pooled together; k-fold
splits it internally. The `test_processed.csv` rows are sealed and only used at the very end.

In [ ]:
import os, json, numpy as np, pandas as pd
import tensorflow as tf
import keras_tuner as kt
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve, matthews_corrcoef)
sns.set_style("whitegrid")

DATA_DIR="."
for base in [".","Task2_DataPrep","../Task2_DataPrep","Airline Satisfaction/Task2_DataPrep"]:
    if os.path.exists(os.path.join(base,"train_processed.csv")): DATA_DIR=base; break
tr=pd.read_csv(os.path.join(DATA_DIR,"train_processed.csv"))
va=pd.read_csv(os.path.join(DATA_DIR,"val_processed.csv"))
te=pd.read_csv(os.path.join(DATA_DIR,"test_processed.csv"))
dev=pd.concat([tr,va],ignore_index=True)                 # development set for k-fold
Xdev=dev.drop(columns="satisfaction").values.astype("float32"); ydev=dev["satisfaction"].values
NFEAT=Xdev.shape[1]
# subsample for the SEARCH only (rankings are stable with less data); final model uses full data
Xs,_,ys,_=train_test_split(Xdev,ydev,train_size=50000,stratify=ydev,random_state=42)
print("development rows:",len(Xdev),"| search subsample:",len(Xs),"| sealed test rows:",len(te))

---
## Step 2 — The MLP builder with a 9-hyperparameter search space

Nadam is fixed as the optimiser; its learning rate, beta_1, beta_2 and epsilon are tunable
so their importance can be measured. The network's depth, width, dropout and activation vary too.

In [ ]:
def build_model(hp):
    lr  = hp.Float("learning_rate", 1e-4, 1e-2, sampling="log")   # most important knob
    n   = hp.Int("num_layers", 1, 3)                              # depth
    units = hp.Choice("units", [32, 64, 128, 256])               # width
    act = hp.Choice("activation", ["relu", "tanh", "elu"])
    dr  = hp.Float("dropout", 0.0, 0.4, step=0.1)                # regularisation
    b1  = hp.Float("beta_1", 0.85, 0.95, step=0.01)             # Nadam internals (checked, not assumed)
    b2  = hp.Float("beta_2", 0.99, 0.9999, sampling="log")
    eps = hp.Float("epsilon", 1e-8, 1e-6, sampling="log")
    m = tf.keras.Sequential([tf.keras.layers.Input((NFEAT,))])
    for _ in range(n):
        m.add(tf.keras.layers.Dense(units, activation=act))
        if dr > 0: m.add(tf.keras.layers.Dropout(dr))
    m.add(tf.keras.layers.Dense(1, activation="sigmoid"))
    m.compile(optimizer=tf.keras.optimizers.Nadam(learning_rate=lr, beta_1=b1, beta_2=b2, epsilon=eps),
              loss="binary_crossentropy", metrics=["accuracy"])
    return m

---
## Step 3 — A cross-validation tuner

Keras Tuner scores each trial on a single split by default. We override `run_trial` so each
trial's score is the **mean k-fold validation accuracy** — the rigorous part. The same mixin
gives us a Random and a Bayesian version sharing identical folds.

In [ ]:
FOLDS=3; MAX_TRIALS=15; MAX_EPOCHS=20; PATIENCE=3     # report used 28 trials; reduced here for runtime

class CVMixin:
    def run_trial(self, trial, X, y):
        hp=trial.hyperparameters
        bs=hp.Choice("batch_size", [128,256,512])
        skf=StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)   # identical folds for fairness
        scores=[]
        for tri,vai in skf.split(X,y):
            tf.keras.utils.set_random_seed(0)
            model=self.hypermodel.build(hp)
            es=tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
            h=model.fit(X[tri],y[tri], validation_data=(X[vai],y[vai]),
                        epochs=MAX_EPOCHS, batch_size=bs, verbose=0, callbacks=[es])
            scores.append(float(max(h.history["val_accuracy"])))
        return {"val_accuracy": float(np.mean(scores))}

class RandomCV(CVMixin, kt.RandomSearch): pass
class BayesCV (CVMixin, kt.BayesianOptimization): pass

---
## Step 4 — Stage 1: race Random Search against Bayesian Optimisation

Same space, same budget, same folds, so the contest is fair.

In [ ]:
def run_search(cls, name):
    t=cls(hypermodel=build_model, objective=kt.Objective("val_accuracy","max"),
          max_trials=MAX_TRIALS, seed=1, overwrite=True, directory="kt_dir", project_name=name)
    t.search(Xs, ys)
    trials=[{"id":int(i),"score":float(tr_.score),"hp":dict(tr_.hyperparameters.values)}
            for i,tr_ in t.oracle.trials.items() if tr_.score is not None]
    trials.sort(key=lambda d:d["id"])
    return trials, t.get_best_hyperparameters(1)[0].values

rand_trials, rand_best = run_search(RandomCV, "random")
bayes_trials, bayes_best = run_search(BayesCV, "bayesian")
print("Random  best CV acc: %.3f%%" % (max(t["score"] for t in rand_trials)*100))
print("Bayesian best CV acc: %.3f%%" % (max(t["score"] for t in bayes_trials)*100))

---
## Step 5 — Convergence graph (best score found so far vs trial)

A flat tail means the budget was long enough; the higher line is the better searcher.

In [ ]:
fig,ax=plt.subplots(figsize=(10,5.5))
for trials,name,c in [(rand_trials,"Random","#4C72B0"),(bayes_trials,"Bayesian","#55A868")]:
    s=np.array([t["score"] for t in trials])*100
    ax.plot(np.arange(1,len(s)+1), np.maximum.accumulate(s), color=c, lw=2.3,
            label=f"{name} (best {np.maximum.accumulate(s)[-1]:.2f}%)")
    ax.scatter(np.arange(1,len(s)+1), s, color=c, s=18, alpha=0.35)
ax.set_xlabel("Trial"); ax.set_ylabel("Best CV accuracy (%)"); ax.legend()
ax.set_title("Stage 1 — search convergence", fontweight="bold")
plt.tight_layout(); plt.savefig("tune_convergence.png", dpi=140, bbox_inches="tight"); plt.show()
winner = "Bayesian" if max(t["score"] for t in bayes_trials) >= max(t["score"] for t in rand_trials) else "Random"
print("Winning searcher:", winner)

---
## Step 5b — Per-hyperparameter exploration (value vs score)

A second reading of the same trials: each hyperparameter's value against the accuracy it
reached. Only the learning rate should show a clear trend; the rest are flat clouds.

In [ ]:
trials_all = rand_trials + bayes_trials
specs=[("learning_rate","Learning rate",True),("dropout","Dropout",False),("num_layers","Layers",False),
       ("units","Units",False),("activation","Activation",None),("batch_size","Batch size",False),
       ("beta_1","beta_1",False),("beta_2","beta_2",True),("epsilon","epsilon",True)]
fig,axes=plt.subplots(3,3,figsize=(13,10))
for ax,(key,label,logx) in zip(axes.ravel(),specs):
    xv=[t["hp"][key] for t in trials_all]; yv=[t["score"]*100 for t in trials_all]   # local names, don't clobber Xs/ys
    if logx is None:
        cats=["relu","tanh","elu"]; xv=[cats.index(v) for v in xv]; ax.set_xticks(range(3)); ax.set_xticklabels(cats)
    ax.scatter(xv,yv,color="#55A868",s=26,alpha=0.7)
    if logx: ax.set_xscale("log")
    ax.set_title(label,fontweight="bold",fontsize=9); ax.set_ylabel("CV acc (%)")
plt.tight_layout(); plt.savefig("tune_slices.png", dpi=135, bbox_inches="tight"); plt.show()

---
## Step 6 — Stage 2: hyperparameter importance

We fit a random-forest surrogate that predicts each trial's score from its hyperparameters,
then read off which ones explain the most variation. Learning rate should dominate; the
Nadam internals should sit at the bottom, confirming their defaults are fine.

In [ ]:
trials = rand_trials + bayes_trials
def feat(hp):
    row=[np.log10(hp["learning_rate"]), hp["num_layers"], hp["units"], hp["dropout"],
         hp["beta_1"], np.log10(hp["beta_2"]), np.log10(max(hp["epsilon"],1e-9)), hp["batch_size"]]
    row += [1.0 if hp["activation"]==a else 0.0 for a in ["relu","tanh","elu"]]
    return row
X=np.array([feat(t["hp"]) for t in trials]); y=np.array([t["score"] for t in trials])
names=["learning_rate","num_layers","units","dropout","beta_1","beta_2","epsilon","batch_size","act_relu","act_tanh","act_elu"]
rf=RandomForestRegressor(n_estimators=400, random_state=42).fit(X,y)
imp={}
for n,v in zip(names, rf.feature_importances_):
    k="activation" if n.startswith("act_") else n
    imp[k]=imp.get(k,0)+v
tot=sum(imp.values()); imp={k:v/tot*100 for k,v in imp.items()}
order=sorted(imp, key=imp.get)
colors=["#C44E52" if k in {"beta_1","beta_2","epsilon"} else "#4C72B0" for k in order]
fig,ax=plt.subplots(figsize=(9,5.5)); ax.barh(order,[imp[k] for k in order],color=colors)
for i,k in enumerate(order): ax.text(imp[k]+0.4,i,f"{imp[k]:.1f}%",va="center",fontsize=9)
ax.set_xlabel("Relative importance (%)"); ax.set_title("Stage 2 — hyperparameter importance", fontweight="bold")
plt.tight_layout(); plt.savefig("tune_importance.png", dpi=140, bbox_inches="tight"); plt.show()
print({k:round(imp[k],1) for k in reversed(order)})

---
## Step 7 — Stage 3: focused search on the influential knobs

Fix the unimportant knobs (activation, batch size, Nadam internals) at defaults; keep learning
rate, dropout and shape open. The learning-rate range is widened upward because the Stage 1
winner sat at the ceiling. Uses the winning searcher (Bayesian) at 5-fold.

In [ ]:
FOLDS=5; MAX_TRIALS_F=12
FIXED=dict(activation="relu", batch_size=256, beta_1=0.9, beta_2=0.999, epsilon=1e-7)
def build_focused(hp):
    lr=hp.Float("learning_rate", 0.003, 0.03, sampling="log")
    n=hp.Int("num_layers", 2, 3); units=hp.Choice("units",[128,256]); dr=hp.Float("dropout",0.0,0.2,step=0.1)
    m=tf.keras.Sequential([tf.keras.layers.Input((NFEAT,))])
    for _ in range(n):
        m.add(tf.keras.layers.Dense(units, activation=FIXED["activation"]))
        if dr>0: m.add(tf.keras.layers.Dropout(dr))
    m.add(tf.keras.layers.Dense(1, activation="sigmoid"))
    m.compile(optimizer=tf.keras.optimizers.Nadam(learning_rate=lr, beta_1=FIXED["beta_1"],
              beta_2=FIXED["beta_2"], epsilon=FIXED["epsilon"]), loss="binary_crossentropy", metrics=["accuracy"])
    return m
class FocusBayes(kt.BayesianOptimization):
    def run_trial(self, trial, X, y):
        hp=trial.hyperparameters; skf=StratifiedKFold(FOLDS, shuffle=True, random_state=42); sc=[]
        for tri,vai in skf.split(X,y):
            tf.keras.utils.set_random_seed(0); mm=self.hypermodel.build(hp)
            es=tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
            h=mm.fit(X[tri],y[tri], validation_data=(X[vai],y[vai]), epochs=25, batch_size=256, verbose=0, callbacks=[es])
            sc.append(float(max(h.history["val_accuracy"])))
        return {"val_accuracy": float(np.mean(sc))}
ft=FocusBayes(hypermodel=build_focused, objective=kt.Objective("val_accuracy","max"),
              max_trials=MAX_TRIALS_F, seed=7, overwrite=True, directory="kt_dir", project_name="focus")
ft.search(Xs, ys)
best={**FIXED, **ft.get_best_hyperparameters(1)[0].values}
print("Best focused config:", best)

# focused-search convergence (5-fold): a firmer estimate, not a performance boost
fscore=np.array([tr_.score for tr_ in ft.oracle.trials.values() if tr_.score is not None])*100
fig,ax=plt.subplots(figsize=(9,4.8))
ax.plot(np.arange(1,len(fscore)+1), np.maximum.accumulate(fscore), color="#55A868", lw=2.3,
        label=f"Bayesian 5-fold (best {np.maximum.accumulate(fscore)[-1]:.2f}%)")
ax.scatter(np.arange(1,len(fscore)+1), fscore, color="#55A868", s=20, alpha=0.4)
ax.set_xlabel("Trial"); ax.set_ylabel("Best CV accuracy (%)"); ax.legend()
ax.set_title("Stage 3 — focused five-fold search convergence", fontweight="bold")
plt.tight_layout(); plt.savefig("tune_focus_convergence.png", dpi=140, bbox_inches="tight"); plt.show()

---
## Step 8 — Verify every fixed setting

Sweep each of the five fixed knobs (activation, batch size, beta_1, beta_2, epsilon) around its
kept value. Flat lines within noise prove fixing it cost nothing; if a knob is not flat (as
activation is), that confirms the kept value was the best of the options.

In [ ]:
def build_cfg(cfg):
    m=tf.keras.Sequential([tf.keras.layers.Input((NFEAT,))])
    for _ in range(cfg["num_layers"]):
        m.add(tf.keras.layers.Dense(cfg["units"], activation=cfg["activation"]))
        if cfg["dropout"]>0: m.add(tf.keras.layers.Dropout(cfg["dropout"]))
    m.add(tf.keras.layers.Dense(1, activation="sigmoid"))
    m.compile(optimizer=tf.keras.optimizers.Nadam(learning_rate=cfg["learning_rate"], beta_1=cfg["beta_1"],
              beta_2=cfg["beta_2"], epsilon=cfg["epsilon"]), loss="binary_crossentropy", metrics=["accuracy"]); return m
def cv3(cfg):
    skf=StratifiedKFold(3, shuffle=True, random_state=42); sc=[]
    for tri,vai in skf.split(Xs,ys):
        tf.keras.utils.set_random_seed(0); mm=build_cfg(cfg)
        es=tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
        h=mm.fit(Xs[tri],ys[tri], validation_data=(Xs[vai],ys[vai]), epochs=20,
                 batch_size=int(cfg["batch_size"]), verbose=0, callbacks=[es])   # honour the swept batch size
        sc.append(max(h.history["val_accuracy"]))
    return np.mean(sc)*100, np.std(sc)*100

sweeps={"activation":["relu","tanh","elu"],"batch_size":[128,256,512],"beta_1":[0.88,0.9,0.92],
        "beta_2":[0.99,0.999,0.9999],"epsilon":[1e-8,1e-7,1e-6]}
fig,axes=plt.subplots(1,5,figsize=(16,4))
for ax,(knob,vals) in zip(axes,sweeps.items()):
    ms=[cv3({**best,knob:v}) for v in vals]
    ax.errorbar(range(len(vals)),[m for m,_ in ms],yerr=[s for _,s in ms],marker="o",capsize=4,color="#333")
    ax.set_xticks(range(len(vals))); ax.set_xticklabels([str(v) for v in vals],rotation=15,fontsize=8)
    ax.set_title(knob,fontweight="bold",fontsize=9)
axes[0].set_ylabel("3-fold CV acc (%)")
plt.tight_layout(); plt.savefig("tune_fixedknob.png", dpi=140, bbox_inches="tight"); plt.show()

---
## Step 9 — Final tuned model: train and evaluate on the sealed test set

Trained on the same training rows as the Task 1 baseline (so the change is due to tuning, not
extra data), early-stopped on the validation set, and scored on the untouched test set.

In [ ]:
Xtr=tr.drop(columns="satisfaction").values.astype("float32"); ytr=tr["satisfaction"].values
Xva=va.drop(columns="satisfaction").values.astype("float32"); yva=va["satisfaction"].values
Xte=te.drop(columns="satisfaction").values.astype("float32"); yte=te["satisfaction"].values
tf.keras.utils.set_random_seed(42)
final=build_cfg(best)
es=tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
final.fit(Xtr,ytr, validation_data=(Xva,yva), epochs=60, batch_size=best["batch_size"], verbose=0, callbacks=[es])
proba=final.predict(Xte,verbose=0).ravel(); pred=(proba>=0.5).astype(int)
tn,fp,fn,tp=confusion_matrix(yte,pred).ravel()
print("TUNED model on the sealed test set:")
print("  Accuracy    %.2f%%" % (accuracy_score(yte,pred)*100))
print("  Precision   %.2f%%" % (precision_score(yte,pred)*100))
print("  Recall      %.2f%%" % (recall_score(yte,pred)*100))
print("  Specificity %.2f%%" % (tn/(tn+fp)*100))
print("  F1-score    %.2f%%" % (f1_score(yte,pred)*100))
print("  MCC         %.3f"   % matthews_corrcoef(yte,pred))
print("  ROC-AUC     %.3f"   % roc_auc_score(yte,proba))

# confusion matrix at the 0.5 cut-off (probability >= 0.5 => "satisfied")
import seaborn as sns
cm=confusion_matrix(yte,pred); lab=["Not satisfied","Satisfied"]
fig,ax=plt.subplots(figsize=(5.5,4.5))
sns.heatmap(cm,annot=True,fmt=",d",cmap="Blues",cbar=False,xticklabels=lab,yticklabels=lab,ax=ax)
ax.set_title("Tuned model — confusion matrix (test set)",fontweight="bold")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.savefig("tune_confusion.png", dpi=140, bbox_inches="tight"); plt.show()

---
## Conclusion

Bayesian optimisation beat random search on the same budget, the learning rate carried almost
all of the importance (the Nadam internals sat at the bottom, so their defaults were kept), and
the focused five-fold search produced a final model that improved on the baseline across six of
seven metrics on the sealed test set, with a train-to-test gap under one percent (no overfitting).
This tuned MLP is the final model for Task 3.